# Binance

Descarga la ultima copia del dataset historico de Kaggle, guarda las tablas base en `datos/` y exporta seis series compactas para dashboard con formato `fecha,valor`.

In [1]:
from pathlib import Path

import kagglehub
import pandas as pd
from kagglehub import KaggleDatasetAdapter

In [2]:
DATASET = "andreschirinos/p2p-bob-exchange"
ARCHIVO_FUENTE = "advice.parquet"
PROPORCION_TRAMO_COMPETITIVO = 0.10
DIAS_SPARKLINE = 7
DIRECTORIO_SALIDA = Path.cwd() / "datos"
DIRECTORIO_SALIDA.mkdir(exist_ok=True)

In [3]:
kg = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    DATASET,
    ARCHIVO_FUENTE,
)

kg.shape

100%|██████████| 40.8M/40.8M [00:27<00:00, 1.56MB/s]


(2141575, 84)

In [4]:
columnas = [
    "timestamp",
    "asset",
    "tradetype",
    "fiatunit",
    "price",
    "tradablequantity",
    "advertiser_userno",
    "minsingletransamount",
    "maxsingletransamount",
]

# En Kaggle/Binance, BUY y SELL estan definidos desde la perspectiva del anunciante.
# Para nuestros indicadores, invertimos ese mapeo para reflejar compra/venta de dolares.
compras = (
    kg.loc[(kg["asset"] == "USDT") & (kg["tradetype"] == "SELL"), columnas]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

ventas = (
    kg.loc[(kg["asset"] == "USDT") & (kg["tradetype"] == "BUY"), columnas]
    .sort_values("timestamp")
    .reset_index(drop=True)
)

compras.to_parquet(DIRECTORIO_SALIDA / "buy.parquet", index=False)
ventas.to_parquet(DIRECTORIO_SALIDA / "sell.parquet", index=False)

print(f"guardado: {DIRECTORIO_SALIDA / 'buy.parquet'}")
print(f"guardado: {DIRECTORIO_SALIDA / 'sell.parquet'}")

guardado: /home/m/Projects/economia/interno/indicadores/tipodecambio/datos/buy.parquet
guardado: /home/m/Projects/economia/interno/indicadores/tipodecambio/datos/sell.parquet


In [5]:
def _calcular_vwap(tabla):
    pesos = tabla["tradablequantity"]
    total = pesos.sum()
    if total == 0:
        return None
    return (tabla["price"] * pesos).sum() / total


def _obtener_tramo_competitivo(tabla, lado):
    cuantila = tabla["price"].quantile(
        PROPORCION_TRAMO_COMPETITIVO if lado == "buy" else 1 - PROPORCION_TRAMO_COMPETITIVO
    )
    if lado == "buy":
        return tabla[tabla["price"] <= cuantila]
    return tabla[tabla["price"] >= cuantila]


def _calcular_vwap_transado(tabla):
    montos = tabla.pivot_table(
        index="timestamp",
        columns="advertiser_userno",
        values="tradablequantity",
        aggfunc="last",
    ).sort_index()
    precios = tabla.pivot_table(
        index="timestamp",
        columns="advertiser_userno",
        values="price",
        aggfunc="last",
    ).sort_index()
    pesos = (-montos.diff()).clip(lower=0)
    total = pesos.sum().sum()
    if total == 0:
        return None
    return (pesos * precios).sum().sum() / total


def _construir_metricas_diarias(tabla, lado):
    tabla = tabla.copy()
    tabla["timestamp"] = pd.to_datetime(tabla["timestamp"])
    tabla = tabla.sort_values("timestamp")
    dia_actual = tabla["timestamp"].max().date()
    filas = []

    for dia, grupo in tabla.groupby(tabla["timestamp"].dt.date):
        tramo_competitivo = _obtener_tramo_competitivo(grupo, lado)
        fila = {
            "fecha": pd.Timestamp(dia),
            "precio_competitivo": _calcular_vwap(tramo_competitivo),
            "precio_profundidad": _calcular_vwap(grupo),
            "precio_transado_estimado": _calcular_vwap_transado(grupo),
            "es_dia_actual": dia == dia_actual,
            "timestamp_reciente": grupo["timestamp"].max(),
        }
        filas.append(fila)

    resultado = pd.DataFrame(filas)
    dias_cerrados = resultado.loc[~resultado["es_dia_actual"]].tail(DIAS_SPARKLINE - 1).copy()
    dia_en_curso = resultado.loc[resultado["es_dia_actual"]].copy()
    if not dia_en_curso.empty:
        dia_en_curso.loc[:, "fecha"] = dia_en_curso["timestamp_reciente"]
    serie_compacta = pd.concat([dias_cerrados, dia_en_curso], ignore_index=True)
    return serie_compacta[["fecha", "precio_competitivo", "precio_profundidad", "precio_transado_estimado"]]


def _exportar_metrica(tabla, lado, metrica):
    salida = tabla[["fecha", metrica]].rename(columns={metrica: "valor"}).copy()
    salida["fecha"] = pd.to_datetime(salida["fecha"]).dt.strftime("%Y-%m-%dT%H:%M:%S")
    ruta = DIRECTORIO_SALIDA / f"binance_{lado}_{metrica}.csv"
    salida.to_csv(ruta, index=False)
    return ruta

In [6]:
rutas_exportadas = []

for lado, tabla in {"buy": compras, "sell": ventas}.items():
    serie_compacta = _construir_metricas_diarias(tabla, lado)
    for metrica in [
        "precio_competitivo",
        "precio_profundidad",
        "precio_transado_estimado",
    ]:
        rutas_exportadas.append(str(_exportar_metrica(serie_compacta, lado, metrica)))

rutas_exportadas

['/home/m/Projects/economia/interno/indicadores/tipodecambio/datos/binance_buy_precio_competitivo.csv',
 '/home/m/Projects/economia/interno/indicadores/tipodecambio/datos/binance_buy_precio_profundidad.csv',
 '/home/m/Projects/economia/interno/indicadores/tipodecambio/datos/binance_buy_precio_transado_estimado.csv',
 '/home/m/Projects/economia/interno/indicadores/tipodecambio/datos/binance_sell_precio_competitivo.csv',
 '/home/m/Projects/economia/interno/indicadores/tipodecambio/datos/binance_sell_precio_profundidad.csv',
 '/home/m/Projects/economia/interno/indicadores/tipodecambio/datos/binance_sell_precio_transado_estimado.csv']

In [5]:
pd.read_csv(DIRECTORIO_SALIDA / "binance_buy_precio_competitivo.csv")

,fecha,valor
0,2026-04-30T00:00:00,9.981232
1,2026-05-01T00:00:00,9.887822
2,2026-05-02T00:00:00,9.881163
3,2026-05-03T00:00:00,9.801212
4,2026-05-04T00:00:00,9.881331
5,2026-05-05T00:00:00,9.961943
6,2026-05-06T18:24:28,9.977962
